<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
# Required packages for this notebook.
# They are imported below; no installation command is needed when they are
# already available in the Jupyter environment.


In [2]:
import re
import unicodedata
from io import StringIO

import pandas as pd
import requests
from bs4 import BeautifulSoup

pd.set_option("display.max_columns", None)


and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def clean_text(value):
    """Return compact, citation-free text from an HTML element or string."""
    if value is None:
        return None

    if hasattr(value, "get_text"):
        text = value.get_text(" ", strip=True)
    else:
        text = str(value).strip()

    text = re.sub(r"\[[^\]]*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text or None


def date_time(table_cell):
    """Extract the launch date and UTC time from a date/time table cell."""
    parts = [
        clean_text(part)
        for part in table_cell.stripped_strings
        if clean_text(part)
    ]

    date_value = parts[0].rstrip(",") if parts else None
    time_value = parts[1] if len(parts) > 1 else None
    return [date_value, time_value]


def booster_version(table_cell):
    """Extract and clean a booster-version value."""
    return clean_text(table_cell)


def landing_status(table_cell):
    """Extract and clean the booster-landing result."""
    return clean_text(table_cell)


def get_mass(table_cell):
    """Extract payload mass in kilograms as text."""
    text = unicodedata.normalize("NFKD", table_cell.get_text(" ", strip=True))
    text = re.sub(r"\[[^\]]*\]", "", text)
    match = re.search(r"([\d,]+(?:\.\d+)?)\s*kg", text, flags=re.IGNORECASE)

    if match:
        return f"{match.group(1)} kg"

    cleaned = re.sub(r"\s+", " ", text).strip()
    return cleaned or None


def extract_column_from_header(header_cell):
    """Extract a clean variable name from a table-header cell."""
    header_copy = BeautifulSoup(str(header_cell), "html.parser")

    for unwanted in header_copy.find_all(["sup"]):
        unwanted.decompose()

    text = header_copy.get_text(" ", strip=True)
    text = re.sub(r"\[[^\]]*\]", "", text)
    text = re.sub(r"\s+", " ", text).strip()

    if not text or text.isdigit():
        return None

    return text


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
ibm_static_url = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/"
    "spacex_web_scraped.html"
)

wikipedia_snapshot_url = (
    "https://en.wikipedia.org/w/index.php?"
    "title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"
)

static_url = ibm_static_url

headers = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/126.0 Safari/537.36"
    )
}


Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [5]:
# TASK 1: Request the Falcon 9 launch page.
# The IBM snapshot is attempted first because it preserves the course data.
# The archived Wikipedia snapshot is used automatically as a fallback.

response = None
request_errors = []

for candidate_url in (ibm_static_url, wikipedia_snapshot_url):
    try:
        candidate_response = requests.get(
            candidate_url,
            headers=headers,
            timeout=60
        )
        candidate_response.raise_for_status()

        if "<table" not in candidate_response.text.lower():
            raise ValueError("The response did not contain an HTML table.")

        response = candidate_response
        static_url = candidate_url
        break

    except (requests.RequestException, ValueError) as error:
        request_errors.append(f"{candidate_url}: {error}")

if response is None:
    raise RuntimeError(
        "The launch page could not be downloaded from either source.\n"
        + "\n".join(request_errors)
    )

print("Page downloaded successfully.")
print("Source:", static_url)
print("HTTP status code:", response.status_code)
print("Downloaded characters:", len(response.text))


Page downloaded successfully.
Source: https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922
HTTP status code: 200
Downloaded characters: 3270197


Create a `BeautifulSoup` object from the HTML `response`


In [6]:
# Create a BeautifulSoup object from the response text.
soup = BeautifulSoup(response.text, "html.parser")

print("BeautifulSoup object created successfully.")


BeautifulSoup object created successfully.


Print the page title to verify if the `BeautifulSoup` object was created properly 


In [7]:
# Verify the parsed page title.
page_title = soup.title.get_text(" ", strip=True) if soup.title else "No title found"
print(page_title)


List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [8]:
# Locate all HTML tables on the page.
html_tables = soup.find_all("table")

print("Number of HTML tables found:", len(html_tables))

if not html_tables:
    raise ValueError("No HTML tables were found on the downloaded page.")


Number of HTML tables found: 25


Starting from the third table is our target table contains the actual launch records.


In [9]:
# Identify the first launch-record table by its expected header names.
first_launch_table = None

for table in html_tables:
    header_text = clean_text(table.find("tr")) or ""
    if (
        "Flight" in header_text
        and "Payload" in header_text
        and "Launch" in header_text
    ):
        first_launch_table = table
        break

if first_launch_table is None:
    raise ValueError("A Falcon launch-record table could not be identified.")

print("Target launch table identified successfully.")
print(clean_text(first_launch_table.find("tr")))


Target launch table identified successfully.
Flight No. Date and time ( UTC ) Version, Booster Launch site Payload Payload mass Orbit Customer Launch outcome Booster landing


You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [10]:
# TASK 2: Extract the column names from the launch-table header.
column_names = []

for header_cell in first_launch_table.find_all("th"):
    name = extract_column_from_header(header_cell)

    if name and name not in column_names:
        column_names.append(name)

print("Extracted column names:")
print(column_names)


Extracted column names:
['Flight No.', 'Date and time ( UTC )', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']


Check the extracted column names


In [11]:
print(column_names)

expected_concepts = [
    "Flight", "Date", "Booster", "Launch site", "Payload",
    "Payload mass", "Orbit", "Customer", "Launch outcome",
    "Booster landing"
]

missing_concepts = [
    concept
    for concept in expected_concepts
    if not any(concept.lower() in name.lower() for name in column_names)
]

if missing_concepts:
    print("Note: some headers use alternative wording:", missing_concepts)
else:
    print("All expected launch-table concepts were found.")


['Flight No.', 'Date and time ( UTC )', 'Version, Booster', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome', 'Booster landing']
All expected launch-table concepts were found.


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [12]:
launch_dict = {
    "Flight No.": [],
    "Date": [],
    "Time": [],
    "Version Booster": [],
    "Launch site": [],
    "Payload": [],
    "Payload mass": [],
    "Orbit": [],
    "Customer": [],
    "Launch outcome": [],
    "Booster landing": [],
}

print("Empty launch dictionary created.")


Empty launch dictionary created.


Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [13]:
# TASK 3: Parse all launch-record tables into launch_dict.
extracted_row = 0
skipped_rows = 0

candidate_tables = []

for table in html_tables:
    header = clean_text(table.find("tr")) or ""
    if (
        "Flight" in header
        and "Payload" in header
        and "Launch" in header
    ):
        candidate_tables.append(table)

for table in candidate_tables:
    for row_element in table.find_all("tr"):
        header_cell = row_element.find("th")
        data_cells = row_element.find_all("td")

        if header_cell is None or not data_cells:
            continue

        flight_text = clean_text(header_cell)

        if not flight_text:
            continue

        flight_match = re.search(r"\d+", flight_text)
        if not flight_match:
            continue

        # A valid launch row should contain at least nine data cells.
        if len(data_cells) < 9:
            skipped_rows += 1
            continue

        try:
            flight_number = flight_match.group()
            launch_date, launch_time = date_time(data_cells[0])

            booster = booster_version(data_cells[1])
            launch_site = clean_text(data_cells[2])
            payload = clean_text(data_cells[3])
            payload_mass = get_mass(data_cells[4])
            orbit = clean_text(data_cells[5])
            customer = clean_text(data_cells[6])
            launch_outcome = clean_text(data_cells[7])
            booster_landing = landing_status(data_cells[8])

            values = {
                "Flight No.": flight_number,
                "Date": launch_date,
                "Time": launch_time,
                "Version Booster": booster,
                "Launch site": launch_site,
                "Payload": payload,
                "Payload mass": payload_mass,
                "Orbit": orbit,
                "Customer": customer,
                "Launch outcome": launch_outcome,
                "Booster landing": booster_landing,
            }

            for key, value in values.items():
                launch_dict[key].append(value)

            extracted_row += 1

        except Exception:
            # Skip an isolated malformed row without corrupting column lengths.
            skipped_rows += 1

print("Launch rows extracted:", extracted_row)
print("Malformed rows skipped:", skipped_rows)

column_lengths = {
    key: len(values)
    for key, values in launch_dict.items()
}

print("Column lengths:", column_lengths)

if extracted_row == 0:
    raise ValueError("No launch records were extracted.")

if len(set(column_lengths.values())) != 1:
    raise ValueError("The parsed columns do not all have the same length.")


Launch rows extracted: 124
Malformed rows skipped: 0
Column lengths: {'Flight No.': 124, 'Date': 124, 'Time': 124, 'Version Booster': 124, 'Launch site': 124, 'Payload': 124, 'Payload mass': 124, 'Orbit': 124, 'Customer': 124, 'Launch outcome': 124, 'Booster landing': 124}


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [14]:
# Create the final Pandas DataFrame.
df = pd.DataFrame(launch_dict)

# Convert Flight No. to a nullable numeric type where possible.
df["Flight No."] = pd.to_numeric(df["Flight No."], errors="coerce").astype("Int64")

# Remove accidental duplicate launch rows while preserving order.
df = df.drop_duplicates().reset_index(drop=True)

print(f"Final web-scraped dataset: {df.shape[0]} rows and {df.shape[1]} columns")
df.head(10)


Final web-scraped dataset: 124 rows and 11 columns


,Flight No.,Date,Time,Version Booster,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Booster landing
0,1,4 June 2010,18:45,F9 v1.0 B0003.1,"CCAFS , SLC-40",Dragon Spacecraft Qualification Unit,NaN,LEO,SpaceX,Success,Failure (parachute)
1,2,8 December 2010,15:43,F9 v1.0 B0004.1,"CCAFS , SLC-40",Dragon demo flight C1 (Dragon C101),NaN,LEO ( ISS ),NASA ( COTS ) NRO,Success,Failure (parachute)
2,3,22 May 2012,07:44,F9 v1.0 B0005.1,"CCAFS , SLC-40",Dragon demo flight C2+ (Dragon C102),525 kg,LEO ( ISS ),NASA ( COTS ),Success,No attempt
3,4,8 October 2012,00:35,F9 v1.0 B0006.1,"CCAFS , SLC-40",SpaceX CRS-1 (Dragon C103),"4,700 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt
4,5,1 March 2013,15:10,F9 v1.0 B0007.1,"CCAFS , SLC-40",SpaceX CRS-2 (Dragon C104),"4,877 kg",LEO ( ISS ),NASA ( CRS ),Success,No attempt
5,6,29 September 2013,16:00,F9 v1.1 B1003,"VAFB , SLC-4E",CASSIOPE,500 kg,Polar orbit LEO,MDA,Success,Uncontrolled (ocean)
6,7,3 December 2013,22:41,F9 v1.1 B1004,"CCAFS , SLC-40",SES-8,"3,170 kg",GTO,SES,Success,No attempt
7,8,6 January 2014,22:06,F9 v1.1,"CCAFS , SLC-40",Thaicom 6,"3,325 kg",GTO,Thaicom,Success,No attempt
8,9,18 April 2014,19:25,F9 v1.1,"Cape Canaveral , LC-40",SpaceX CRS-3 (Dragon C105),"2,296 kg",LEO ( ISS ),NASA ( CRS ),Success,Controlled (ocean)
9,10,14 July 2014,15:15,F9 v1.1,"Cape Canaveral , LC-40",Orbcomm-OG2 -1 (6 satellites),"1,316 kg",LEO,Orbcomm,Success,Controlled (ocean)


We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


In [15]:
# Validate and export the web-scraped dataset.
required_columns = list(launch_dict.keys())
missing_columns = [
    column for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

if df.empty:
    raise ValueError("The final DataFrame is empty.")

output_file = "spacex_web_scraped.csv"
df.to_csv(output_file, index=False)

print(f"Saved {output_file} successfully.")
print("Dataset shape:", df.shape)
print("\nMissing values by column:")
print(df.isna().sum())


Saved spacex_web_scraped.csv successfully.
Dataset shape: (124, 11)

Missing values by column:
Flight No.         0
Date               0
Time               0
Version Booster    0
Launch site        0
Payload            0
Payload mass       2
Orbit              0
Customer           0
Launch outcome     0
Booster landing    0
dtype: int64


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
